In [4]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
from arabert.preprocess import ArabertPreprocessor
import os

# ==========================================
# 1. CẤU HÌNH VÀ TIỀN XỬ LÝ (ARABERT BASE-TWITTER)
# ==========================================
MODEL_NAME = "aubmindlab/bert-base-arabertv02-twitter"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STANCE2ID = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL = {0: "Against", 1: "Favor", 2: "None"}
SENTIMENT2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
SARCASM2ID = {"No": 0, "Yes": 1}

# Khởi tạo Preprocessor chính chủ
arabert_prep = ArabertPreprocessor(model_name="bert-base-arabertv02-twitter")

def clean_arabic_tweet(text):
    if not isinstance(text, str): return ""
    return arabert_prep.preprocess(text)

def load_data(file_path, is_train=True):
    df = pd.read_csv(file_path, keep_default_na=False)
    for col in ["target", "text"]:
        df[col] = df[col].astype(str).str.strip()
    df["clean_text"] = df["text"].apply(clean_arabic_tweet)
    
    if is_train:
        df["label_stance"] = df["stance"].map(STANCE2ID).fillna(2).astype(int)
        df["label_sentiment"] = df["sentiment"].map(SENTIMENT2ID).fillna(1).astype(int)
        df["label_sarcasm"] = df["sarcasm"].map(SARCASM2ID).fillna(0).astype(int)
    return df

print("Đang load dữ liệu...")
train_df = load_data("../data/train.csv", is_train=True)
dev_df = load_data("../data/dev.csv", is_train=False)

def tokenize_func(examples):
    tokenized = tokenizer(
        examples["target"], 
        examples["clean_text"], 
        padding="max_length", 
        truncation=True, \
        max_length=128
    )
    if "label_stance" in examples:
        tokenized["labels_stance"] = examples["label_stance"]
        tokenized["labels_sentiment"] = examples["label_sentiment"]
        tokenized["labels_sarcasm"] = examples["label_sarcasm"]
    return tokenized

# ==========================================\n# 2. KIẾN TRÚC MULTI-TASK & TRAINER CHUẨN HOÁ
# ==========================================
class MultiTaskAraBERT(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.stance_head = nn.Linear(hidden_size, 3)
        self.sentiment_head = nn.Linear(hidden_size, 3)
        self.sarcasm_head = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(cls_output)
        return self.stance_head(pooled), self.sentiment_head(pooled), self.sarcasm_head(pooled)

class MultiTaskTrainer(Trainer):
    def __init__(self, stance_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stance_weights = stance_weights 

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_stance = inputs.pop("labels_stance")
        labels_sentiment = inputs.pop("labels_sentiment")
        labels_sarcasm = inputs.pop("labels_sarcasm")
        
        logits_stance, logits_sentiment, logits_sarcasm = model(**inputs)
        
        loss_fct_stance = nn.CrossEntropyLoss(weight=self.stance_weights.to(logits_stance.device), label_smoothing=0.1)
        loss_fct_sentiment = nn.CrossEntropyLoss()
        loss_fct_sarcasm = nn.CrossEntropyLoss()
        
        loss_stance = loss_fct_stance(logits_stance, labels_stance)
        loss_sentiment = loss_fct_sentiment(logits_sentiment, labels_sentiment)
        loss_sarcasm = loss_fct_sarcasm(logits_sarcasm, labels_sarcasm)
        
        total_loss = loss_stance + 0.2 * loss_sentiment + 0.1 * loss_sarcasm
        
        # FIX CHUẨN: Return Tuple Logits để HuggingFace Unpack mượt mà
        if return_outputs:
            return total_loss, (logits_stance, logits_sentiment, logits_sarcasm)
        else:
            return total_loss

def compute_metrics(eval_pred):
    logits_tuple = eval_pred.predictions
    labels_tuple = eval_pred.label_ids

    if isinstance(logits_tuple, (tuple, list)):
        logits = logits_tuple[0] 
    elif isinstance(logits_tuple, dict):
         logits = logits_tuple["logits_stance"] # Đã sửa NameError
    else:
        logits = logits_tuple
        
    if isinstance(labels_tuple, (tuple, list)):
        labels = labels_tuple[0]
    else:
        labels = labels_tuple

    logits = np.array(logits)
    labels = np.array(labels)
    
    if logits.ndim > 2: 
         logits = logits.reshape(-1, logits.shape[-1])
         labels = labels.flatten()

    preds = np.argmax(logits, axis=-1)
    f_ag = f1_score(labels, preds, labels=[0], average="macro")
    f_fav = f1_score(labels, preds, labels=[1], average="macro")
    
    return {"Favg2": (f_fav + f_ag) / 2.0}

# ==========================================
# 3. QUY TRÌNH 10-FOLD CV THIẾT LẬP LẠI VÀ TRÍCH OOF
# ==========================================
print("\n--- BẮT ĐẦU 10-FOLD CROSS VALIDATION VỚI ARABERT ---")
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train_df), 3))

# Stratify theo tổ hợp Target + Stance chống lệch fold
strat_key = train_df["target"] + "_" + train_df["stance"]

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, strat_key)):
    print(f"\n🚀 ĐANG HUẤN LUYỆN FOLD {fold + 1}/{N_SPLITS}...")
    
    fold_train_df = train_df.iloc[train_idx]
    fold_val_df = train_df.iloc[val_idx]
    
    stance_labels = fold_train_df["label_stance"].tolist()
    class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(stance_labels), y=stance_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
    
    cols_data = ["target", "clean_text", "label_stance", "label_sentiment", "label_sarcasm"]
    fold_train_ds = Dataset.from_pandas(fold_train_df[cols_data]).map(tokenize_func, batched=True).remove_columns(cols_data)
    fold_val_ds = Dataset.from_pandas(fold_val_df[cols_data]).map(tokenize_func, batched=True).remove_columns(cols_data)
    
    model = MultiTaskAraBERT(MODEL_NAME)
    
    args = TrainingArguments(
        output_dir=f"../model/arabert_fold_{fold}",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=6, 
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        weight_decay=0.01,
        bf16=True, 
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="Favg2",
        greater_is_better=True,
        label_names=["labels_stance", "labels_sentiment", "labels_sarcasm"],
        logging_steps=50,
    )
    
    trainer = MultiTaskTrainer(
        stance_weights=class_weights_tensor,
        model=model, args=args, train_dataset=fold_train_ds, eval_dataset=fold_val_ds,
        data_collator=data_collator, compute_metrics=compute_metrics
    )
    
    trainer.train()
    torch.save(model.state_dict(), f"../model/arabert_fold_{fold}.pt")
    
    val_loader = DataLoader(
        fold_val_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids"]), 
        batch_size=16, shuffle=False
    )
    
    model.eval().to(device)
    fold_val_probs = []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids=batch["input_ids"].to(device), 
                attention_mask=batch["attention_mask"].to(device),
                token_type_ids=batch["token_type_ids"].to(device)
            )
            probs = F.softmax(outputs[0], dim=-1)
            fold_val_probs.append(probs.cpu().numpy())
    
    oof_probs[val_idx] = np.vstack(fold_val_probs)
    del model, trainer
    torch.cuda.empty_cache()

np.save("../model/arabert_oof_probs.npy", oof_probs)
print("\n✅ Đã lưu phiên bản OOF mới tại: '../model/arabert_oof_probs.npy'")

# ==========================================
# 4. INFERENCE LÊN TẬP TEST (DEV SET)
# ==========================================
print("\n--- INFERENCE TẬP TEST (DEV SET) BẰNG 10 MODELS ---")
cols_test = ["target", "clean_text"]
test_ds = Dataset.from_pandas(dev_df[cols_test]).map(tokenize_func, batched=True).remove_columns(cols_test)
test_loader = DataLoader(
    test_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids"]), 
    batch_size=16, shuffle=False
)

all_fold_test_probs = []
for fold in range(N_SPLITS):
    print(f"Đang inference Test bằng AraBERT Fold {fold + 1}...")
    model = MultiTaskAraBERT(MODEL_NAME)
    model.load_state_dict(torch.load(f"../model/arabert_fold_{fold}.pt", map_location=device, weights_only=True))
    model.eval().to(device)
    
    fold_probs = []
    with torch.no_grad():
        for batch in test_loader:
            outputs = model(
                input_ids=batch["input_ids"].to(device), 
                attention_mask=batch["attention_mask"].to(device),
                token_type_ids=batch["token_type_ids"].to(device)
            )
            probs = F.softmax(outputs[0], dim=-1)
            fold_probs.append(probs.cpu().numpy())
            
    all_fold_test_probs.append(np.vstack(fold_probs))
    del model
    torch.cuda.empty_cache()

final_probs = np.mean(all_fold_test_probs, axis=0)
np.save("../model/arabert_test_probs.npy", final_probs)
print("✅ Đã lưu phiên bản Test Probs mới tại: '../model/arabert_test_probs.npy'")

# ==========================================
# 5. DÒ MỊN MARGIN THRESHOLD VÀ IN ĐIỂM FAVG2 CHO ARABERT
# ==========================================
print("\n=== GIAI ĐOẠN 5: DÒ THRESHOLD MARGIN SIÊU MỊN CHO ARABERT ===")
true_train_labels = train_df["label_stance"].values
true_dev_labels = dev_df["stance"].map(STANCE2ID).fillna(2).astype(int).values

best_favg2 = 0
best_thresholds = {"Against": 0.33, "Favor": 0.33}

for th_against in np.arange(0.2, 0.7, 0.02):
    for th_favor in np.arange(0.2, 0.7, 0.02):
        custom_preds = []
        for p in oof_probs:
            margin = p - np.array([th_against, th_favor, 0.0])
            custom_preds.append(2 if margin.max() < 0 else int(np.argmax(margin)))
                
        f_ag = f1_score(true_train_labels, custom_preds, labels=[0], average="macro")
        f_fav = f1_score(true_train_labels, custom_preds, labels=[1], average="macro")
        favg2 = (f_fav + f_ag) / 2.0
        
        if favg2 > best_favg2:
            best_favg2 = favg2
            best_thresholds = {"Against": th_against, "Favor": th_favor}

print(f"🏆 Điểm OOF Favg2 AraBERT Đạt Được: {best_favg2:.4f}")
print(f"Bộ ngưỡng tối ưu AraBERT -> Against: {best_thresholds['Against']:.2f}, Favor: {best_thresholds['Favor']:.2f}")

final_preds_arabert = []
for p in final_probs:
    margin = p - np.array([best_thresholds['Against'], best_thresholds['Favor'], 0.0])
    final_preds_arabert.append(2 if margin.max() < 0 else int(np.argmax(margin)))

dev_f_ag = f1_score(true_dev_labels, final_preds_arabert, labels=[0], average="macro")
dev_f_fav = f1_score(true_dev_labels, final_preds_arabert, labels=[1], average="macro")
print(f"📊 ĐIỂM FAVG2 ARABERT TRÊN DEV SET (ƯỚC TÍNH LEADERBOARD): {(dev_f_ag + dev_f_fav) / 2.0:.4f}")

Đang load dữ liệu...

--- BẮT ĐẦU 10-FOLD CROSS VALIDATION VỚI ARABERT ---

🚀 ĐANG HUẤN LUYỆN FOLD 1/10...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3618.99it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2

Epoch,Training Loss,Validation Loss,Favg2
1,1.262547,1.070915,0.404261
2,1.086377,1.071730,0.466674
3,0.925766,0.984617,0.492649
4,0.824630,0.987403,0.498482
5,0.753181,1.014895,0.521745
6,0.720446,1.009548,0.512381



🚀 ĐANG HUẤN LUYỆN FOLD 2/10...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3663.39it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2

...

Đang inference Test bằng AraBERT Fold 9...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5457.80it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đang inference Test bằng AraBERT Fold 10...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4680.16it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Đã lưu phiên bản Test Probs mới tại: '../model/arabert_test_probs.npy'

=== GIAI ĐOẠN 5: DÒ THRESHOLD MARGIN SIÊU MỊN CHO ARABERT ===
🏆 Điểm OOF Favg2 AraBERT Đạt Được: 0.8042
Bộ ngưỡng tối ưu AraBERT -> Against: 0.24, Favor: 0.20
📊 ĐIỂM FAVG2 ARABERT TRÊN DEV SET (ƯỚC TÍNH LEADERBOARD): 0.7923
